In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm

# node2vec
from node2vec import Node2Vec

# clustering
from sklearn.cluster import KMeans, DBSCAN, OPTICS

# deep stuff
import torch
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader

/home/jann/GAT/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.community_results import community_metrics
from src.clustering import label_propagation, louvain, leiden
from src.DGIModel import DGIModel

# Load dataset

In [3]:
data_path = './data/deezer_clean_data/'

edges = os.path.join(data_path, 'HR_edges.csv')
genres = os.path.join(data_path, 'HR_genres.json')

In [4]:
edges_df = pd.read_csv(edges)
g = nx.from_pandas_edgelist(edges_df, source='node_1', target='node_2')

In [5]:
with open(genres, 'r') as f:
    genres_data = json.load(f)


In [6]:
def encode_genres(genre_list, unique_genres):
    encoding = [1 if genre in genre_list else 0 for genre in unique_genres]
    return encoding


def decode_genres(genre_vector, unique_genres):
    decoded_genres = [unique_genres[i] for i in range(len(genre_vector)) if genre_vector[i] == 1]
    return decoded_genres

In [7]:
all_genres = set()
for genres in genres_data.values():
    all_genres.update(genres)

all_genres = sorted(all_genres)

for user_id, genres in genres_data.items():
    if isinstance(user_id, str):
        user_id = int(user_id)
    if user_id in g.nodes:
        g.nodes[user_id]['genres'] = encode_genres(genres, all_genres)

In [8]:
sel_graph = 'deezer'
# nx.draw(g, with_labels=True)

# Apply graph based clustering

## Label Propagation

In [9]:
lp_labels = label_propagation(g)

# plot_communities(g, lp_labels, title=f"Label Propagation {sel_graph}")

# Calculate metrics for Louvain
lp_metrics = community_metrics(g, lp_labels)
lp_metrics

{'Number of Communities': 916,
 'Average Community Size': 59.577510917030565,
 'Modularity': 0.6737606352463761,
 'Coverage': 0.71361817094271,
 'Performance': 0.9460924050525683}

## Louvain

In [10]:
lv_labels = louvain(g)

# plot_communities(g, lv_labels, title=f"Louvain {sel_graph}")

# Calculate metrics for Louvain
lv_metrics = community_metrics(g, lv_labels)
lv_metrics

{'Number of Communities': 24,
 'Average Community Size': 2273.875,
 'Modularity': 0.7378400014177648,
 'Coverage': 0.8040674264655702,
 'Performance': 0.9293519480033885}

## Leiden

In [11]:
ld_labels = leiden(g)

# plot_communities(g, ld_labels, title=f"Leiden {sel_graph}")

# Calculate metrics for Louvain
ld_metrics = community_metrics(g, ld_labels)
ld_metrics

{'Number of Communities': 26,
 'Average Community Size': 2098.9615384615386,
 'Modularity': 0.7403265397796991,
 'Coverage': 0.8065082034997852,
 'Performance': 0.9284314944127493}

# Apply GAT Embedding

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [13]:
data = from_networkx(g)
data.x = torch.tensor([g.nodes[n]['genres'] for n in g.nodes], dtype=torch.float)
# data = data.to(device)

In [14]:
from src.tune import tune

best_p = tune(data)
best_p

[I 2024-08-11 18:40:12,768] A new study created in memory with name: no-name-7f1f247b-3e50-435c-bf32-18005982dfcd


In [15]:
hidden_channels = 64
out_channels = 32
num_layers = 3
heads = 16
dropout = 0.15
lr = 1e-3
l2_reg = 5e-4
epochs = 100

In [16]:
train_loader = DataLoader([data], batch_size=16, shuffle=True)

In [17]:
model = DGIModel(in_channels=data.num_features, hidden_channels=hidden_channels, out_channels=out_channels, num_layers=num_layers, heads=heads, dropout=dropout)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2_reg)

accumulation_steps = 4

def train():
    for i, data in enumerate(train_loader):
        model.train()
        optimizer.zero_grad()
        pos_z, neg_z, summary = model(data)
        loss = model.dgi.loss(pos_z, neg_z, summary)
        loss = loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

    return loss.item()


for epoch in tqdm(range(epochs)):
    loss = train()
    # if epoch % 10 == 0:
    print(f'Epoch {epoch}, Loss: {loss}')


  1%|          | 1/100 [00:12<20:40, 12.53s/it]

Epoch 0, Loss: 0.3475322127342224


  2%|▏         | 2/100 [00:25<21:12, 12.99s/it]

Epoch 1, Loss: 0.6069443225860596


  3%|▎         | 3/100 [00:39<21:11, 13.11s/it]

...

Epoch 97, Loss: 0.2904663383960724


 99%|█████████▉| 99/100 [21:32<00:13, 13.04s/it]

Epoch 98, Loss: 0.2920321226119995


100%|██████████| 100/100 [21:45<00:00, 13.06s/it]

Epoch 99, Loss: 0.2837904393672943


In [18]:
# Extract node embeddings
model.eval()
with torch.no_grad():
    node_embeddings = model.encoder(data.x, data.edge_index).detach().cpu().numpy()

In [19]:
model_path = 'deezer.torch'
torch.save(model.state_dict(), model_path)

In [20]:
# model = DGIModel(in_channels=data.num_features, hidden_channels=hidden_channels, out_channels=out_channels, num_layers=num_layers, heads=heads, dropout=dropout)
# model.load_state_dict(torch.load(model_path))
# model.eval()

In [21]:
best_eps = 0
best_mod = -1

# for i in tqdm(np.arange(0.3, 0.7, 0.1)):
#     dbscan = DBSCAN(eps=i, min_samples=5)
#     communities = dbscan.fit_predict(emb)
#     mod = community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))['Modularity']
#     if mod > best_mod:
#         best_eps, best_mod = i, mod

optics = OPTICS(min_samples=5)
communities = optics.fit_predict(node_embeddings)
# plot_communities(g, communities, title=f"GAT {sel_graph}")
community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))

{'Number of Communities': 857,
 'Average Community Size': 63.67911318553092,
 'Modularity': -0.00011071075613328701,
 'Coverage': 0.7942400873541254,
 'Performance': 0.2083772623359983}

In [25]:
best_n = 950
best_mod = -1

# for i in tqdm(np.arange(12, 48, 1)):
#     kmeans = KMeans(n_clusters=i)
#     communities = kmeans.fit_predict(node_embeddings)
#     mod = community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))['Modularity']
#     if mod > best_mod:
#         best_n, best_mod = i, mod

kmeans = KMeans(n_clusters=best_n)
communities = kmeans.fit_predict(node_embeddings)

# plot_communities(g, communities, title=f"GAT {sel_graph}")
community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))

{'Number of Communities': 950,
 'Average Community Size': 57.445263157894736,
 'Modularity': -0.0001268534697850763,
 'Coverage': 0.0019570375068747215,
 'Performance': 0.9976134178971277}